In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 VALIDACIÓN DE FRECUENCIA CORREGIDA (DOPPLER + DERIVA) EN DATOS REALES LIGO
 ================================================================================
 Para cada evento GW, calcula la frecuencia esperada (deriva + Doppler cósmico)
 y aplica el pipeline UAT de RMS normalizado usando una banda pasante centrada
 en dicha frecuencia. Compara con la banda fija 172‑260 Hz.

 Requiere: gwpy, astropy, numpy, scipy, matplotlib, pandas, requests.
 Ejecutar en Colab o entorno con acceso a internet.
 ================================================================================
 Autor: Miguel Ángel Percudani
 ================================================================================
"""

import numpy as np
import pandas as pd
import io, os, gc, warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# --- Dependencias (gwpy) ---
try:
    from gwpy.timeseries import TimeSeries
    from astropy.time import Time
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gwpy"])
    from gwpy.timeseries import TimeSeries
    from astropy.time import Time

import requests
from scipy.signal import butter, filtfilt, iirnotch

# =============================================================================
# 1. PARÁMETROS UAT / DOPPLER
# =============================================================================
F_BASE = 84.4               # Hz (referencia 2023-05-27)
FECHA_REF = datetime(2023, 5, 27)
ALPHA_F = 0.046              # Hz/día

V_DETECTOR = 600.0           # km/s hacia el Gran Atractor
C_LUZ = 299792.458           # km/s
# Dirección del movimiento (Gran Atractor)
RA_GA = 157.5
DEC_GA = -46.0
# Dirección de la señal (Cáncer) – triangulación SVD
RA_SENAL = 124.78
DEC_SENAL = 7.85

# Banda empírica original (para comparación)
LOW_BAND_EMPIRICAL = 172.0
HIGH_BAND_EMPIRICAL = 260.0

# Notches de línea eléctrica
NOTCH_FREQS = [60, 120, 180, 240, 300, 360, 420, 480, 540, 600, 660, 720, 780, 840]

# Parámetros del pipeline RMS
ATTRACTOR_APRIL_2026 = 0.7071
DATE_APRIL_2026 = Time('2026-04-23', scale='utc')
DATE_MAY_2026    = Time('2026-05-10', scale='utc')
ALPHA_AMPLITUDE = (0.7086 - 0.7071) / (DATE_MAY_2026 - DATE_APRIL_2026).jd

WINDOW_SEC = 1.0
TOLERANCE = 0.05
DETECTION_THRESHOLD = 50.0

# =============================================================================
# 2. FUNCIONES DE CÁLCULO DE FRECUENCIA CORREGIDA
# =============================================================================
def angulo_entre_vectores(ra1, dec1, ra2, dec2):
    ra1r, dec1r = np.radians(ra1), np.radians(dec1)
    ra2r, dec2r = np.radians(ra2), np.radians(dec2)
    cos_theta = (np.sin(dec1r) * np.sin(dec2r) +
                 np.cos(dec1r) * np.cos(dec2r) * np.cos(ra2r - ra1r))
    return cos_theta

# Coseno del ángulo señal‑movimiento (constante)
cos_theta = angulo_entre_vectores(RA_SENAL, DEC_SENAL, RA_GA, DEC_GA)
# Factor Doppler fijo
DOPPLER_FACTOR = 1.0 + (V_DETECTOR / C_LUZ) * cos_theta

def frecuencia_corregida(fecha_evento):
    """Devuelve la frecuencia observada (deriva + Doppler) para una fecha dada."""
    delta_dias = (fecha_evento - FECHA_REF).days
    f_base = F_BASE + ALPHA_F * delta_dias
    return f_base * DOPPLER_FACTOR

# =============================================================================
# 3. FUNCIONES DEL PIPELINE RMS
# =============================================================================
def expected_attractor(gps_time):
    t = Time(gps_time, format='gps', scale='utc')
    days = (t - DATE_APRIL_2026).jd
    return ATTRACTOR_APRIL_2026 + ALPHA_AMPLITUDE * days

def apply_notches(data, fs):
    y = data.copy()
    nyq = fs / 2.0
    for f0 in NOTCH_FREQS:
        if f0 < nyq - 1:
            b, a = iirnotch(f0, Q=30, fs=fs)
            y = filtfilt(b, a, y)
    return y

def bandpass_filter(data, fs, lowcut, highcut, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def analyse_segment(data, gps_centre, lowcut, highcut):
    """RMS normalizado en ventanas de 1 s, con banda pasante dinámica."""
    fs = data.sample_rate.value
    # Aplicar notches y filtro pasabanda personalizado
    notched = apply_notches(data.value, fs)
    filtered = bandpass_filter(notched, fs, lowcut, highcut)
    samp_win = int(WINDOW_SEC * fs)
    n_windows = len(filtered) // samp_win
    attractor = expected_attractor(gps_centre)
    hits = 0
    for i in range(n_windows):
        seg = filtered[i*samp_win : (i+1)*samp_win]
        rms = np.sqrt(np.mean(seg**2))
        peak = np.max(np.abs(seg))
        rms_norm = rms / peak if peak > 0 else 0.0
        if abs(rms_norm - attractor) < TOLERANCE:
            hits += 1
    return 100.0 * hits / n_windows if n_windows > 0 else 0.0

# =============================================================================
# 4. DESCARGA DE CATÁLOGO Y ANÁLISIS
# =============================================================================
print("=" * 70)
print(" VALIDACIÓN DE FRECUENCIA DOPPLER EN EVENTOS GW REALES")
print("=" * 70)
print(f"Factor Doppler aplicado : {DOPPLER_FACTOR:.6f} (blueshift)")
print(f"Banda empírica (fija)   : {LOW_BAND_EMPIRICAL:.0f}‑{HIGH_BAND_EMPIRICAL:.0f} Hz")
print(f"Banda dinámica centrada : en f_corregida ± 10 Hz")
print("=" * 70)

# Obtener catálogo GWTC
try:
    url = "https://gwosc.org/eventapi/csv/GWTC/"
    df = pd.read_csv(io.StringIO(requests.get(url).text), skipinitialspace=True)
    events = [{'name': row['commonName'], 'gps': row['GPS']}
              for _, row in df.iterrows() if row['GPS'] > 1126259462]
    print(f"Catálogo descargado: {len(events)} eventos.\n")
except Exception as e:
    print(f"Error de catálogo: {e}")
    events = [{"name": "GW150914", "gps": 1126259462.4},
              {"name": "GW170817", "gps": 1187008882.4},
              {"name": "GW190814", "gps": 1249852257.0}]

# Analizar primeros 30 eventos (para demo)
resultados = []
for ev in events[:30]:
    gps = ev['gps']
    name = ev['name']
    t_gps = Time(gps, format='gps', scale='utc')
    fecha = t_gps.datetime  # datetime de Python
    f_corr = frecuencia_corregida(fecha)

    # Definir banda dinámica centrada en f_corr (ancho ±10 Hz)
    low_dyn = max(1.0, f_corr - 10.0)
    high_dyn = f_corr + 10.0

    try:
        data = TimeSeries.fetch_open_data('H1', gps-16, gps+16, verbose=False)

        # 1. Usar banda empírica fija
        hit_fixed = analyse_segment(data, gps, LOW_BAND_EMPIRICAL, HIGH_BAND_EMPIRICAL)
        # 2. Usar banda dinámica (centrada en frecuencia corregida)
        hit_dyn = analyse_segment(data, gps, low_dyn, high_dyn)

        resultados.append((name, gps, fecha, f_corr, hit_fixed, hit_dyn))
        print(f"{name}: f_corr={f_corr:.1f} Hz -> fixed={hit_fixed:.1f}% | dynamic={hit_dyn:.1f}%")
    except Exception as e:
        print(f"{name}: descarga fallida ({e})")

# =============================================================================
# 5. COMPARACIÓN
# =============================================================================
print("\n" + "=" * 70)
print(" RESUMEN COMPARATIVO")
print("=" * 70)
det_fixed = sum(1 for r in resultados if r[4] > DETECTION_THRESHOLD)
det_dyn   = sum(1 for r in resultados if r[5] > DETECTION_THRESHOLD)
print(f"Detecciones con banda fija (172‑260 Hz): {det_fixed}/{len(resultados)}")
print(f"Detecciones con banda dinámica (centrada en f_corr): {det_dyn}/{len(resultados)}")
if det_dyn > det_fixed:
    print("→ La banda centrada en la frecuencia teórica mejora la detección.")
elif det_dyn < det_fixed:
    print("→ La banda empírica aún supera a la dinámica; puede deberse al ancho elegido (±10 Hz).")
else:
    print("→ Ambas bandas dan resultados similares.")
print("=" * 70)

 VALIDACIÓN DE FRECUENCIA DOPPLER EN EVENTOS GW REALES
Factor Doppler aplicado : 1.000962 (blueshift)
Banda empírica (fija)   : 172‑260 Hz
Banda dinámica centrada : en f_corregida ± 10 Hz
Catálogo descargado: 219 eventos.

GW150914: descarga fallida (filter critical frequencies must be greater than 0)
GW151012: descarga fallida (filter critical frequencies must be greater than 0)
GW151226: descarga fallida (filter critical frequencies must be greater than 0)
GW170104: descarga fallida (filter critical frequencies must be greater than 0)
GW170608: descarga fallida (filter critical frequencies must be greater than 0)
GW170729: descarga fallida (filter critical frequencies must be greater than 0)
GW170809: descarga fallida (filter critical frequencies must be greater than 0)
GW170814: descarga fallida (filter critical frequencies must be greater than 0)
GW170817: descarga fallida (filter critical frequencies must be greater than 0)
GW170818: descarga fallida (filter critical frequencies m